# CN x1.1 — Complete Formal Backtest

This notebook reproduces the user-directed promotion of **CN x1.1** from the frozen candidate evidence. It covers the exact 102 non-overlapping 10-session rebalances from 2022-07-01, retains all 252 target-position rows, reconstructs 372 weight-change transactions, and reports the certified provider cutoff of 2026-08-03.

Research boundary: `research_only=true`; `trade_ready=false`.


## Model contract

- Parent score: frozen `r0_cn_x1_0_raw_return_rank` / `current_cn_ohlcv`
- Sector breadth: mean percentile of each sector's Top 3 names
- Risk-on: Top 4 sectors, Top 1 name per sector, 25% each
- Risk-off: 100% CSI300 fallback
- Gate: at least 2 of MA200 trend, 60-session momentum, and CN130 MA60 breadth
- Rebalance: every 10 provider sessions
- Execution delay: one session
- Cost: 20 bps per unit of turnover

Frozen authorization: workflow `31022910416`, artifact `8937409026`.


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
EVIDENCE = ROOT / "data/research/cn_x1_1_regime_gated_candidate_v1"
FORMAL = ROOT / "data/research/formal_backtests/cn_x1_1.json"
CONFIG = ROOT / "configs/models/cn_x1_1.yaml"
assert EVIDENCE.is_dir() and FORMAL.is_file() and CONFIG.is_file()


In [ ]:
evidence_manifest = json.loads((EVIDENCE / "evidence_manifest.json").read_text())
for item in evidence_manifest["files"]:
    path = EVIDENCE / item["path"]
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    assert observed == item["sha256"], item["path"]
print("Frozen evidence hashes verified:", len(evidence_manifest["files"]))


In [ ]:
decision = json.loads((EVIDENCE / "decision.json").read_text())
assert decision["candidate_authorized"] is True
assert decision["decision"] == "cn_x1_1_regime_gated_candidate_authorized"
assert all(decision["gates"].values())
decision


In [ ]:
periods = pd.read_csv(EVIDENCE / "rebalance_periods.csv")
holdings = pd.read_csv(EVIDENCE / "holdings.csv")
half_year = pd.read_csv(EVIDENCE / "half_year_results.csv")
formal = json.loads(FORMAL.read_text())

assert len(periods) == 102
assert len(holdings) == 252
assert len(formal["trades"]) == 372
assert formal["model_id"] == "cn_x1_1"
assert formal["evidence_completeness"]["status"] == "complete"
formal["metrics"]


In [ ]:
trace = pd.DataFrame(formal["report"])
trace["date"] = pd.to_datetime(trace["date"])
trace[["date", "account", "bench_hs300", "drawdown", "risk_state"]].tail()


In [ ]:
ax = trace.plot(x="date", y=["account", "bench_hs300"], figsize=(11, 5), title="CN x1.1 vs CSI300")
ax.set_ylabel("Growth of 1.0")
plt.show()


In [ ]:
half_year[["window", "total_return", "benchmark_return", "relative_excess", "max_drawdown", "risk_on_share"]]


In [ ]:
position_table = pd.DataFrame(formal["positions"])
trade_table = pd.DataFrame(formal["trades"])
attribution = pd.DataFrame(formal["attribution"])

print("Position rows:", len(position_table))
print("Trade rows:", len(trade_table))
print("Attribution rows:", len(attribution))
position_table.tail(12)


In [ ]:
trade_table.tail(20)


In [ ]:
attribution.sort_values("value", ascending=False).head(20)


In [ ]:
state = trace.groupby("risk_state").agg(
    periods=("date", "count"),
    compounded_return=("period_return", lambda x: (1 + x).prod() - 1),
    compounded_benchmark=("benchmark_return", lambda x: (1 + x).prod() - 1),
    turnover=("turnover", "sum"),
    cost=("transaction_cost", "sum"),
)
state["relative_excess"] = (1 + state["compounded_return"]) / (1 + state["compounded_benchmark"]) - 1
state


## Interpretation

The historical selection interval (2022H2–2025H2) produced +60.60% versus +3.66% for CSI300, or +54.93% compounded relative excess, with a -24.44% maximum drawdown. The reporting-only 2026 extension remained ahead of CSI300 by +2.77% relative, but the combined complete-path maximum drawdown widened to about -37.06%.

The promotion therefore changes the formal publication identity from CN x1.0 to CN x1.1 without changing the frozen rule or reusing 2026 to tune the model.
